# Tracking the Emissions of Sentiment Analysis

**Measuring and benchmarking dictionary sentiment analysis with CodeCarbon**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seankellyhp/eco-measure-amsterdam/blob/main/notebooks/sentiment_tracking.ipynb)

In [ ]:
# Setup. On Colab: install the packages Colab does not have, and download the data.
# Locally (uv sync), everything is already installed and in the repository.
import os
import sys
import urllib.request
import warnings
from pathlib import Path

if "google.colab" in sys.modules:
    %pip install -q -U codecarbon "ecologits[openai]" polars plotnine

# The notebooks read the data from ../data, the data folder of the repository.
for file in ["data/austen_books.csv", "data/bing.csv"]:
    path = Path("..") / file
    if not path.exists():
        path.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/seankellyhp/eco-measure-amsterdam/main/{file}", path)

# Quieten CodeCarbon before the first tracker is created.
os.environ["CODECARBON_LOG_LEVEL"] = "error"
warnings.filterwarnings("ignore")

import plotnine
import polars as pl

plotnine.options.figure_size = (7, 3.5)
plotnine.options.dpi = 110
pl.Config.set_tbl_hide_dataframe_shape(True)

## Description

This notebook continues the Getting Started tutorial. It measures one
sentiment analysis with a `with OfflineEmissionsTracker()` block, and then
benchmarks two ways of writing it against each other with CodeCarbon’s
tasks.

It runs on its own: the setup below loads the packages, the data, and the
helper functions from Getting Started.

## Setup

In [ ]:
import random
import sys
import tempfile
from pathlib import Path

import nltk
import polars as pl
from codecarbon import OfflineEmissionsTracker
from plotnine import aes, geom_jitter, geom_violin, ggplot, labs, theme, theme_minimal

In [ ]:
austen_books = pl.read_csv("../data/austen_books.csv").with_columns(
    pl.col("text").fill_null("")
)

NLTK provides the sentence tokenizer, the word tokenizer, and the second
sentiment dictionary (Bird et al., 2009). NLTK downloads them once, into one of
the folders NLTK searches:

In [ ]:
nltk.download(
    ["punkt_tab", "opinion_lexicon"],
    download_dir=f"{sys.prefix}/nltk_data",
    quiet=True,
)

from nltk.corpus import opinion_lexicon
from nltk.tokenize import sent_tokenize, word_tokenize

Each tracker is created with the same settings, so keep them in one place. A
temporary folder holds the log here; use a folder of your own to keep it
after the session ends:

In [ ]:
log_dir = Path(tempfile.mkdtemp(prefix="codecarbon_results_"))

tracker_args = dict(output_dir=log_dir, country_iso_code="DEU", log_level="error")

Splitting text into words, and into sentences, is needed several times below,
so each gets a function of its own. `unnest_tokens()` is the same as in
Getting Started:

In [ ]:
def unnest_tokens(data, column):
    """Split text into words, one row each, like tidytext's unnest_tokens().

    The text is lower-cased, and a word is a run of letters and digits,
    with an apostrophe inside it kept ("don't").
    """
    return (
        data
        .with_columns(
            pl.col(column)
            .str.to_lowercase()
            .str.extract_all(r"[\p{L}\p{N}]+(?:'[\p{L}]+)*").alias("word")
        )
        .drop(column)
        .explode("word")
        .drop_nulls("word")
    )


def unnest_sentences(data, column):
    """Split text into sentences, one row each, like tidytext's unnest_sentences().

    NLTK's sentence tokenizer finds the sentences, and `id` numbers them
    from 1 in order.
    """
    return (
        data
        .with_columns(
            pl.col(column)
            .map_elements(sent_tokenize, return_dtype=pl.List(pl.String)).alias("sentence")
        )
        .drop(column)
        .explode("sentence")
        .with_row_index("id", offset=1)
        .with_columns(
            pl.col("id")
            .cast(pl.Int64)
        )
    )

Number the chapters of each novel. A new chapter starts at each line that
begins with “Chapter”:

In [ ]:
books = austen_books.with_columns(
    linenumber=pl.int_range(1, pl.len() + 1).over("book"),
    chapter=pl.col("text").str.contains(r"(?i)^chapter [\divxlc]").cum_sum().over("book"),
)

## Comparisons of Simple Sentiment Analysis Approaches

Next we will use two approaches for sentiment analysis, with a focus on
comparing their emissions: the same dictionary method, written with polars
and with NLTK.

### Data Preparation

Sentiment is scored sentence by sentence, for every sentence of
*Pride and Prejudice*. The data has one row per printed line, so first join
each chapter’s lines back into one string. The titles in front of names are
written without their full stop, or some “Mr.” would still end a sentence:

In [ ]:
chapters = (
    books.filter(
        pl.col("book") == "Pride & Prejudice",
        pl.col("chapter") > 0,
        ~pl.col("text").str.contains(r"(?i)^chapter [\divxlc]"),
    )
    .group_by("chapter", maintain_order=True)
    .agg(text=pl.col("text").str.join(" "))
    .with_columns(
        pl.col("text")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
        .str.replace_all(r"\b(Mr|Mrs|Dr)\.", "$1")
    )
)

Then split the chapters into sentences, one row each:

In [ ]:
pride = unnest_sentences(chapters, "text")

pride

### Dictionary Sentiment using polars

The first approach is to split sentences into words, join the Bing lexicon
(Hu & Liu, 2004), and count positive against negative words. A sentence
with no lexicon word, or as many positive as negative ones, is neutral:

In [ ]:
bing = pl.read_csv("../data/bing.csv")


def score_with_dictionary(data):
    scores = (
        unnest_tokens(data.select("id", "sentence"), "sentence")
        .join(bing, on="word")
        .group_by("id")
        .agg(score=pl.when(pl.col("sentiment") == "positive").then(1).otherwise(-1).sum())
    )
    return (
        data.select("id")
        .join(scores, on="id", how="left")
        .select(
            "id",
            sentiment=pl.when(pl.col("score") > 0)
            .then(pl.lit("positive"))
            .when(pl.col("score") < 0)
            .then(pl.lit("negative"))
            .otherwise(pl.lit("neutral")),
        )
    )

Put the call inside a `with OfflineEmissionsTracker()` block. The block can
hold any number of statements, and what they create is still there
afterwards. Here it scores every sentence in the novel:

In [ ]:
with OfflineEmissionsTracker(project_name="dictionary sentiment", **tracker_args) as tracker:
    dictionary_labels = score_with_dictionary(pride)

dict_emit = pl.DataFrame([vars(tracker.final_emissions_data)])

dict_emit.select(
    "project_name",
    "duration",
    energy_wh=pl.col("energy_consumed") * 1000,
    co2e_mg=pl.col("emissions") * 1e6,
)

Labeling all the sentences takes a fraction of a
second. The labels:

In [ ]:
dictionary_labels["sentiment"].value_counts(sort=True)

### Benchmark: polars or NLTK?

The same dictionary approach can be written with NLTK (Bird et al., 2009),
which includes the same sentiment lexicon (`opinion_lexicon`) and its own
sentence and word tokenizers, downloaded in the setup:

In [ ]:
positive_words = set(opinion_lexicon.positive())
negative_words = set(opinion_lexicon.negative())

To compare the two packages fairly, both start from the same texts from
*Pride and Prejudice*, one row per chapter. Both then split the chapters into
sentences, tokenize them and score every sentence.

In [ ]:
def sentiment_polars(texts):
    return score_with_dictionary(unnest_sentences(texts, "text"))


def score_with_nltk(sentences):
    scores = [
        sum((word in positive_words) - (word in negative_words) for word in word_tokenize(sentence.lower()))
        for sentence in sentences
    ]
    return pl.DataFrame({"id": range(1, len(scores) + 1), "score": scores}).select(
        "id",
        sentiment=pl.when(pl.col("score") > 0)
        .then(pl.lit("positive"))
        .when(pl.col("score") < 0)
        .then(pl.lit("negative"))
        .otherwise(pl.lit("neutral")),
    )


def sentiment_nltk(texts):
    sentences = [sentence for text in texts["text"] for sentence in sent_tokenize(text)]
    return score_with_nltk(sentences)

Both split the chapters with NLTK’s sentence tokenizer; only the way they
split sentences into words differs. To compare their labels, score the same
sentences, `pride`, with both. The confusion matrix shows that they
largely agree:

In [ ]:
nltk_labels = score_with_nltk(pride["sentence"])

pl.DataFrame(
    {"polars": dictionary_labels["sentiment"], "nltk": nltk_labels["sentiment"]}
).pivot(on="nltk", index="polars", values="nltk", aggregate_function="len", sort_columns=True).sort("polars")

CodeCarbon has no benchmark of its own, but tasks make one. `carbon_bench()`
runs each function repeatedly, in random order, as one task each, after one
untracked warm-up run of each:

In [ ]:
def carbon_bench(functions, times=20, warmup=True):
    if warmup:
        for function in functions.values():
            function()

    runs = [(name, i) for name in functions for i in range(1, times + 1)]
    random.shuffle(runs)

    tracker = OfflineEmissionsTracker(project_name="carbon bench", **tracker_args)
    rows = []
    try:
        for name, i in runs:
            tracker.start_task(f"{name}_{i}")
            functions[name]()
            task = tracker.stop_task()
            rows.append({"expr": name, "iter": i, **vars(task)})
    finally:
        tracker.stop()

    return pl.DataFrame(rows)

`plot_bench()` draws the runs of each function, one point per run over the
shape of their distribution:

In [ ]:
def plot_bench(bench, metric="energy_consumed"):
    return (
        ggplot(bench, aes("expr", metric, fill="expr"))
        + geom_violin(alpha=0.4)
        + geom_jitter(width=0.1, height=0, alpha=0.7)
        + labs(x="", y=metric, title=f"carbon_bench: {metric}")
        + theme_minimal()
        + theme(legend_position="none")
    )

Compare the two:

In [ ]:
random.seed(1234)

bench = carbon_bench(
    {
        "polars": lambda: sentiment_polars(chapters),
        "nltk": lambda: sentiment_nltk(chapters),
    },
    times=20,
)

bench.group_by("expr", maintain_order=True).agg(
    duration_sec=pl.col("duration").median(),
    energy_wh=pl.col("energy_consumed").median() * 1000,
    co2e_mg=pl.col("emissions").median() * 1e6,
)

In [ ]:
plot_bench(bench, metric="energy_consumed")

## Technical Details

See the [CodeCarbon documentation](https://mlco2.github.io/codecarbon/) and
the [EcoLogits documentation](https://ecologits.ai) for their methods, and
the [TidyCarbon repository](https://github.com/seankellyhp/Tidycarbon) for
the R version of this tutorial.

## References

Bird, S., Klein, E., & Loper, E. (2009). *Natural language processing with Python: Analyzing text with the natural language toolkit*. O’Reilly Media.

Hu, M., & Liu, B. (2004). Mining and summarizing customer reviews. *Proceedings of the Tenth ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 168–177.

## Disclaimer

This method is intended for academic research; ensure legality and ethics
when processing chat data.

## Contact Details